In [62]:
from pathlib import Path
from typing import Union

from ase import Atoms  # type: ignore
from ase.io.lammpsdata import read_lammps_data  # type: ignore
import nglview as nv

In [7]:
atoms = read_lammps_data('geometry.dat', style='full')

nv.show_ase(atoms)


NGLWidget()

In [61]:
import numpy as np
import nglview as nv
from ase import Atoms

# Get positions and symbols
positions = atoms.get_positions()
symbols = atoms.get_chemical_symbols()
# Count total number of water molecules (number of oxygens)
total_waters = sum(1 for sym in symbols if sym == 'O')

# Count sodiums and chlorines
num_sodiums = sum(1 for sym in symbols if sym in ['Na', 'SOD', 'Na+'])
idx_Na = [i for i,s in enumerate(symbols) if s in ('Na')]
print(idx_Na)
num_chlorines = sum(1 for sym in symbols if sym in ['Cl', 'CLA', 'Cl-'])
idx_Cl = [i for i,s in enumerate(symbols) if s in ('Cl', 'CLA', 'Cl-')]



keep = []
num_waters_removed = 0
num_hydrogens_removed = 0
max_z = None
min_z = None

for i, (sym, pos) in enumerate(zip(symbols, positions)):
    z = pos[2]
    if max_z is None or z > max_z:
        max_z = z
    if min_z is None or z < min_z:
        min_z = z
    if sym == 'O' and z > 0:
        num_waters_removed += 1
        continue  # remove all oxygens on z > 0 side
    elif sym == 'H' and z > 0:
        if abs(z) < 0.1:
            keep.append(i)  # keep H if |z| < 0.1
        else:
            num_hydrogens_removed += 1
        # else: remove H if |z| >= 0.1
    else:
        keep.append(i)  # keep all other atoms

filtered_atoms = atoms[keep]

# Calculate box volume in Angstrom^3 and convert to liters
cell = atoms.get_cell()
if hasattr(cell, 'volume'):
    box_volume_A3 = cell.volume
    box_volume_A3 = abs(cell[0][0] * cell[1][1] * 50)
    print(cell)
else:
    box_volume_A3 = np.abs(np.linalg.det(cell))
box_volume_L = box_volume_A3 * 1e-27  # 1 A^3 = 1e-27 L

# Calculate salt mass (NaCl) in grams
# Molar mass: Na = 22.9898 g/mol, Cl = 35.45 g/mol, NaCl = 58.44 g/mol
# Assume 1:1 ratio, so number of NaCl pairs = min(num_sodiums, num_chlorines)
num_nacl = min(num_sodiums, num_chlorines)
NA = 6.02214076e23  # Avogadro's number
mass_nacl_g = num_nacl * 58.44 / NA  # grams

# Salt concentration in g/L
if box_volume_L > 0:
    salt_conc_g_per_L = mass_nacl_g / box_volume_L
else:
    salt_conc_g_per_L = np.nan

print(f"Total number of water molecules (O atoms): {total_waters}")
print(f"Number of sodiums: {num_sodiums}")
print(f"Number of chlorines: {num_chlorines}")
print(f"Salt concentration: {salt_conc_g_per_L:.4f} g/L")
print(f"Number of water molecules removed (O atoms with z > 0): {num_waters_removed}")
print(f"Number of hydrogens removed (H atoms with z > 0 and |z| > 0.1): {num_hydrogens_removed}")
print(f"Maximum z: {max_z}")
print(f"Minimum z: {min_z}")

# Visualize
nv_view = nv.show_ase(filtered_atoms)
nv_view

[6381, 6382, 6383, 6384, 6385, 6386, 6387, 6388, 6389, 6390, 6391, 6392]
Cell([31.9006, 33.813995, 91.199996])
Total number of water molecules (O atoms): 1711
Number of sodiums: 12
Number of chlorines: 12
Salt concentration: 21.5911 g/L
Number of water molecules removed (O atoms with z > 0): 414
Number of hydrogens removed (H atoms with z > 0 and |z| > 0.1): 828
Maximum z: 20.0
Minimum z: -50.0


NGLWidget()